In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Gabungin dataset_final.csv (yang udah di-split train/test oleh split_data.py) dengan
sintesis paraphrase & dummy data, jadi 2 set file training siap pakai:

  - 03_Indobert/Data/3class/train.csv  + test.csv  (skema 3 kelas: positif/netral/negatif)
  - 03_Indobert/Data/binary/train.csv  + test.csv  (skema binary: negatif/non-negatif)

ATURAN ANTI-LEAKAGE (penting):
  - TEST set SELALU cuma baris asli dari dataset_final.csv (split=='test'). Ga pernah
    ditambah sintesis/dummy apapun, dan SAMA PERSIS dipakai di 2 skema (3class & binary),
    biar hasil keduanya bisa dibandingin secara fair.
  - Sintesis paraphrase & dummy cuma ditempel ke TRAIN. Mereka dibuat dari komentar yang
    sudah dipastikan ada di split=='train' sebelumnya (ambil_negatif_train.py), jadi ga ada
    resiko sintesis "lahir" dari baris yang kebetulan masuk test.

KOMPOSISI PER SKEMA:
  - 3class train = train asli + sintesis negatif + dummy negatif + sintesis positif + dummy positif
    (positif ikut diperbanyak karena di 3 kelas dia minoritas vs netral)
  - binary train = train asli + sintesis negatif + dummy negatif SAJA
    (positif TIDAK diikutkan karena di binary dia melebur ke "non-negatif" bareng netral yg
    udah dominan -- nambahin paraphrase positif di binary cuma nambah bias, ga menyelesaikan
    masalah minoritas negatif)

Setiap baris output ditandai kolom 'sumber': asli / sintesis_paraphrase / dummy_baru -- buat
transparansi metodologi (biar di skripsi jelas berapa persen data training yg sintetis).
Semua kolom asli dari tiap file dipertahankan (union kolom), kolom yg ga relevan buat suatu
baris otomatis kosong.

File sintesis/dummy diasumsikan ada langsung di 03_Indobert/Data/ (sintesis_negatif_train.csv,
dummy_negatif_baru.csv, sintesis_positif_train.csv, dummy_positif_baru.csv).

Script ini CUMA BACA dataset_final.csv dan file sintesis/dummy (read-only), outputnya file
BARU di folder 03_Indobert/Data/3class dan /binary. Ga nimpa apapun yg udah ada di path lama."""
from pathlib import Path

import pandas as pd


def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
    )

# --- Path input ---
DATASET_PATH = REPO_ROOT / '01_pengumpulan_data' / '07_persiapan_data' / 'hasil' / 'dataset_final.csv'
SINTESIS_DIR = REPO_ROOT / '03_Indobert' / 'Data'
SINTESIS_NEGATIF_PATH = SINTESIS_DIR / 'sintesis_negatif_train.csv'
DUMMY_NEGATIF_PATH = SINTESIS_DIR / 'dummy_negatif_baru.csv'
SINTESIS_POSITIF_PATH = SINTESIS_DIR / 'sintesis_positif_train.csv'
DUMMY_POSITIF_PATH = SINTESIS_DIR / 'dummy_positif_baru.csv'

# --- Path output ---
OUT_3CLASS_DIR = REPO_ROOT / '03_Indobert' / 'Data' / '3class'
OUT_BINARY_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'binary'


def wajib_ada(path):
    if not path.exists():
        raise FileNotFoundError(f'{path} tidak ditemukan.')
    return path


def load_csv(path):
    return pd.read_csv(wajib_ada(path), dtype=str, keep_default_na=False, encoding='utf-8')


# --- Load dataset_final.csv (sudah ada kolom split/fold dari split_data.py) ---
df = load_csv(DATASET_PATH)

required = {'split', 'label_final', 'label_name', 'comment_text_clean'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Kolom berikut tidak ada di dataset_final.csv: {missing}')

if not (df['split'] == 'train').any() or not (df['split'] == 'test').any():
    raise ValueError(
        "Kolom 'split' belum lengkap keisi 'train'/'test'. Jalankan split_data.py dulu sebelum ini."
    )

df_asli = df.copy()
df_asli['text'] = df_asli['comment_text_clean']
df_asli['sumber'] = 'asli'

train_asli = df_asli[df_asli['split'] == 'train'].reset_index(drop=True)
test_asli = df_asli[df_asli['split'] == 'test'].reset_index(drop=True)

# --- Load sintesis & dummy ---
sintesis_negatif = load_csv(SINTESIS_NEGATIF_PATH)
sintesis_negatif['text'] = sintesis_negatif['paraphrase_text']
sintesis_negatif['sumber'] = 'sintesis_paraphrase'

dummy_negatif = load_csv(DUMMY_NEGATIF_PATH)
dummy_negatif['sumber'] = 'dummy_baru'
# dummy_negatif.csv udah punya kolom 'text' sendiri, ga perlu di-derive

sintesis_positif = load_csv(SINTESIS_POSITIF_PATH)
sintesis_positif['text'] = sintesis_positif['paraphrase_text']
sintesis_positif['sumber'] = 'sintesis_paraphrase'

dummy_positif = load_csv(DUMMY_POSITIF_PATH)
dummy_positif['sumber'] = 'dummy_baru'

# --- Validasi ringan: pastikan sintesis & dummy negatif/positif labelnya konsisten ---
for nama, d, label_expected in [
    ('sintesis_negatif', sintesis_negatif, 'negatif'),
    ('dummy_negatif', dummy_negatif, 'negatif'),
    ('sintesis_positif', sintesis_positif, 'positif'),
    ('dummy_positif', dummy_positif, 'positif'),
]:
    if not (d['label_final'] == label_expected).all():
        raise ValueError(f"File {nama} ada baris dengan label_final != '{label_expected}', cek lagi datanya.")


def gabung(daftar_df):
    """Concat dengan union kolom -- kolom yg ga ada di suatu df otomatis NaN lalu diisi string kosong."""
    hasil = pd.concat(daftar_df, ignore_index=True, sort=False)
    return hasil.fillna('')


# --- 3 KELAS: train = asli + (sintesis & dummy negatif) + (sintesis & dummy positif) ---
train_3class = gabung([train_asli, sintesis_negatif, dummy_negatif, sintesis_positif, dummy_positif])
test_3class = test_asli.copy()

# --- BINARY: train = asli + (sintesis & dummy negatif) SAJA, positif TIDAK diikutkan ---
train_binary = gabung([train_asli, sintesis_negatif, dummy_negatif])
test_binary = test_asli.copy()

# --- Simpan ---
for d in (OUT_3CLASS_DIR, OUT_BINARY_DIR):
    d.mkdir(parents=True, exist_ok=True)

train_3class.to_csv(OUT_3CLASS_DIR / 'train.csv', index=False, encoding='utf-8')
test_3class.to_csv(OUT_3CLASS_DIR / 'test.csv', index=False, encoding='utf-8')
train_binary.to_csv(OUT_BINARY_DIR / 'train.csv', index=False, encoding='utf-8')
test_binary.to_csv(OUT_BINARY_DIR / 'test.csv', index=False, encoding='utf-8')


# --- Laporan ---
def laporan(nama, sub_df, kolom_label):
    print(f'\n{nama} -- total {len(sub_df)} baris')
    print('  per sumber:')
    for sumber, jumlah in sub_df['sumber'].value_counts().items():
        print(f'    {sumber:20s}: {jumlah}')
    print(f'  per {kolom_label}:')
    for label, jumlah in sub_df[kolom_label].value_counts().items():
        print(f'    {label:14s}: {jumlah}')


print('=' * 60)
print('LAPORAN GABUNG FINAL TRAINING (3 KELAS)')
print('=' * 60)
laporan('TRAIN 3class', train_3class, 'label_final')
laporan('TEST 3class', test_3class, 'label_final')

print('\n' + '=' * 60)
print('LAPORAN GABUNG FINAL TRAINING (BINARY)')
print('=' * 60)
laporan('TRAIN binary', train_binary, 'label_name')
laporan('TEST binary', test_binary, 'label_name')

print(f'\nDisimpan ke:')
print(f'  {OUT_3CLASS_DIR / "train.csv"}')
print(f'  {OUT_3CLASS_DIR / "test.csv"}')
print(f'  {OUT_BINARY_DIR / "train.csv"}')
print(f'  {OUT_BINARY_DIR / "test.csv"}')

LAPORAN GABUNG FINAL TRAINING (3 KELAS)

TRAIN 3class -- total 2928 baris
  per sumber:
    asli                : 1668
    sintesis_paraphrase : 1190
    dummy_baru          : 70
  per label_final:
    netral        : 1430
    positif       : 844
    negatif       : 654

TEST 3class -- total 417 baris
  per sumber:
    asli                : 417
  per label_final:
    netral        : 358
    positif       : 33
    negatif       : 26

LAPORAN GABUNG FINAL TRAINING (BINARY)

TRAIN binary -- total 2218 baris
  per sumber:
    asli                : 1668
    sintesis_paraphrase : 520
    dummy_baru          : 30
  per label_name:
    non-negatif   : 1564
    negatif       : 654

TEST binary -- total 417 baris
  per sumber:
    asli                : 417
  per label_name:
    non-negatif   : 391
    negatif       : 26

Disimpan ke:
  c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\3class\train.csv
  c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\3class\test.csv
 

In [3]:
def laporan_perbandingan(nama, df_sebelum, df_sesudah, kolom_label):
    """Bandingin komposisi label SEBELUM (asli doang) vs SESUDAH (+ sintesis & dummy)."""
    print(f'\n{nama}')
    labels = sorted(set(df_sebelum[kolom_label]) | set(df_sesudah[kolom_label]))
    n_before = len(df_sebelum)
    n_after = len(df_sesudah)
    print(f'  {"label":14s} {"SEBELUM (asli)":>20s} {"SESUDAH (+sintesis)":>22s} {"selisih":>10s}')
    for label in labels:
        c_before = int((df_sebelum[kolom_label] == label).sum())
        c_after = int((df_sesudah[kolom_label] == label).sum())
        pct_before = 100 * c_before / n_before if n_before else 0
        pct_after = 100 * c_after / n_after if n_after else 0
        print(f'  {label:14s} {c_before:6d} ({pct_before:5.1f}%)     '
              f'{c_after:6d} ({pct_after:5.1f}%)        +{c_after - c_before}')
    print(f'  {"TOTAL":14s} {n_before:6d} (100.0%)     {n_after:6d} (100.0%)        +{n_after - n_before}')
    
    
print('\n' + '=' * 60)
print('PERBANDINGAN SEBELUM vs SESUDAH AUGMENTASI (cuma bagian TRAIN, TEST ga berubah)')
print('=' * 60)
laporan_perbandingan('TRAIN 3class (label_final)', train_asli, train_3class, 'label_final')
laporan_perbandingan('TRAIN binary (label_name)', train_asli, train_binary, 'label_name')


PERBANDINGAN SEBELUM vs SESUDAH AUGMENTASI (cuma bagian TRAIN, TEST ga berubah)

TRAIN 3class (label_final)
  label                SEBELUM (asli)    SESUDAH (+sintesis)    selisih
  negatif           104 (  6.2%)        654 ( 22.3%)        +550
  netral           1430 ( 85.7%)       1430 ( 48.8%)        +0
  positif           134 (  8.0%)        844 ( 28.8%)        +710
  TOTAL            1668 (100.0%)       2928 (100.0%)        +1260

TRAIN binary (label_name)
  label                SEBELUM (asli)    SESUDAH (+sintesis)    selisih
  negatif           104 (  6.2%)        654 ( 29.5%)        +550
  non-negatif      1564 ( 93.8%)       1564 ( 70.5%)        +0
  TOTAL            1668 (100.0%)       2218 (100.0%)        +550


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Load 4 file hasil gabung_final_training.py (3class & binary, train & test) dan cetak
laporan komposisi lengkap -- dipakai buat mastiin data udah bener sebelum training IndoBERT.

Dicek: jumlah baris, proporsi label, proporsi sumber (asli/sintesis_paraphrase/dummy_baru),
dan statistik panjang teks (jumlah kata) per label -- biar kalau ada komentar yg ganjil
(kosong/kependekan) ketauan sebelum masuk tokenizer."""
from pathlib import Path

import pandas as pd


def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
    )

DATA_DIR = REPO_ROOT / '03_Indobert' / 'Data'
FILES = {
    '3class': {
        'train': DATA_DIR / '3class' / 'train.csv',
        'test': DATA_DIR / '3class' / 'test.csv',
        'label_col': 'label_final',
    },
    'binary': {
        'train': DATA_DIR / 'binary' / 'train.csv',
        'test': DATA_DIR / 'binary' / 'test.csv',
        'label_col': 'label_name',
    },
}


def load(path):
    if not path.exists():
        raise FileNotFoundError(f'{path} tidak ditemukan. Jalankan gabung_final_training.py dulu.')
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8')


def cek_teks_kosong(df, nama):
    kosong = df['text'].str.strip().eq('')
    if kosong.any():
        print(f'  PERHATIAN: {nama} punya {kosong.sum()} baris dengan teks kosong!')


def laporan(nama, df, label_col):
    print(f'\n{"=" * 55}')
    print(f'{nama}  (total {len(df)} baris)')
    print('=' * 55)

    print(f'-- Proporsi {label_col} --')
    for label, jumlah in df[label_col].value_counts().items():
        pct = 100 * jumlah / len(df) if len(df) else 0
        print(f'  {label:14s}: {jumlah:5d}  ({pct:5.1f}%)')

    print('-- Proporsi sumber --')
    for sumber, jumlah in df['sumber'].value_counts().items():
        pct = 100 * jumlah / len(df) if len(df) else 0
        print(f'  {sumber:20s}: {jumlah:5d}  ({pct:5.1f}%)')

    print('-- Panjang teks (jumlah kata) per label --')
    df = df.copy()
    df['_jumlah_kata'] = df['text'].str.split().apply(len)
    stats = df.groupby(label_col)['_jumlah_kata'].agg(['mean', 'min', 'max'])
    for label, row in stats.iterrows():
        print(f'  {label:14s}: rata-rata {row["mean"]:.1f} kata (min {int(row["min"])}, max {int(row["max"])})')

    cek_teks_kosong(df, nama)


for skema, paths in FILES.items():
    df_train = load(paths['train'])
    df_test = load(paths['test'])
    label_col = paths['label_col']

    required = {'text', 'sumber', label_col}
    for nama_df, df in [('train', df_train), ('test', df_test)]:
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f'File {skema}/{nama_df}.csv kurang kolom: {missing}')

    laporan(f'{skema.upper()} - TRAIN', df_train, label_col)
    laporan(f'{skema.upper()} - TEST', df_test, label_col)

print('\nSelesai cek komposisi. Kalau semua wajar (ga ada teks kosong, proporsi sesuai ekspektasi), lanjut ke training.')


3CLASS - TRAIN  (total 2928 baris)
-- Proporsi label_final --
  netral        :  1430  ( 48.8%)
  positif       :   844  ( 28.8%)
  negatif       :   654  ( 22.3%)
-- Proporsi sumber --
  asli                :  1668  ( 57.0%)
  sintesis_paraphrase :  1190  ( 40.6%)
  dummy_baru          :    70  (  2.4%)
-- Panjang teks (jumlah kata) per label --
  negatif       : rata-rata 10.2 kata (min 2, max 43)
  netral        : rata-rata 13.3 kata (min 2, max 172)
  positif       : rata-rata 8.7 kata (min 2, max 92)

3CLASS - TEST  (total 417 baris)
-- Proporsi label_final --
  netral        :   358  ( 85.9%)
  positif       :    33  (  7.9%)
  negatif       :    26  (  6.2%)
-- Proporsi sumber --
  asli                :   417  (100.0%)
-- Panjang teks (jumlah kata) per label --
  negatif       : rata-rata 10.9 kata (min 2, max 54)
  netral        : rata-rata 13.5 kata (min 2, max 148)
  positif       : rata-rata 8.0 kata (min 2, max 27)

BINARY - TRAIN  (total 2218 baris)
-- Proporsi label_name

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Fine-tune IndoBERTweet buat 2 skema (binary & 3class), diadaptasi dari 02_finetune_cv.ipynb
versi lama, disesuaikan sama data hasil gabung_final_training.py (03_Indobert/Data/<skema>/).

PERBEDAAN PENTING vs notebook lama:
  1. Baris sintesis/dummy TIDAK PUNYA kolom 'fold' (cuma di-assign sebelum ada augmentasi).
     Makanya di tiap fold CV, baris yang dijadiin HELD-OUT VALIDATION cuma baris 'asli'
     (pakai kolom fold yg asli), sedangkan baris sintesis/dummy SELALU masuk ke training di
     setiap fold -- supaya metrik CV ngukur kemampuan model di komentar ASLI, bukan di
     kalimat buatan sendiri.
  2. `test.csv` sekarang itu HOLD-OUT SUNGGUHAN yang udah dipisah dari awal (ga pernah
     disentuh augmentasi). Jadi CV (5-fold) di sini cuma buat development/tuning; hasil akhir
     yang dilaporkan di skripsi adalah SATU model final (dilatih pakai SELURUH train.csv)
     dievaluasi SEKALI ke test.csv -- bukan rata-rata 5 fold kaya notebook lama.
  3. class_weights dihitung ULANG dari distribusi train.csv yang baru (bukan pakai angka
     config.json lama yang udah obsolete krn datanya beda).

Model: indolem/indobertweet-base-uncased (sama kaya sebelumnya). max_len=56 (dipertahankan
biar kompatibel/comparable sama baseline S0/S1).

CATATAN SOAL TESTING: bagian yang murni logic (pemisahan fold, hitung class weight, mapping
label) sudah ditest lewat `test_finetune_cv_final.py` tanpa butuh GPU/download model. Bagian
yang beneran panggil IndoBERTweet (tokenizer + training) BELUM bisa ditest di sini karena
environment ini ga ada akses ke huggingface hub (network allowlist) dan ga ada GPU -- jadi
WAJIB kamu coba jalanin sendiri di mesin yang ada GPU + akses internet dulu sebelum dipakai
buat laporan beneran.
"""
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# ============================ KONFIGURASI ============================
SCHEMES = ['binary', '3class']
EPOCHS = 4
LR = 2e-5
BATCH_TRAIN = 16       # turunkan ke 8 kalau CUDA out of memory
BATCH_EVAL = 32
MAX_LEN = 56
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
VAL_SIZE_INTERNAL = 0.1   # dipecah dari bagian "remain" tiap fold, buat early stopping
RUN_CV = False              # False -> skip 5-fold CV, langsung ke model final doang (lebih cepet,
                           # tapi ga dapet metrik CV per fold -- cocok kalau CV-nya udah pernah
                           # dijalanin sebelumnya dan sekarang cuma mau liat train vs test)
SAVE_MODELS = False        # True -> simpan model fold terbaik + model final ke models/<skema>/
SEED = 42
MODEL_NAME = 'indolem/indobertweet-base-uncased'

LABEL2ID = {
    'binary': {'non-negatif': 0, 'negatif': 1},
    '3class': {'negatif': 0, 'netral': 1, 'positif': 2},
}
LABEL_COL = {'binary': 'label_name', '3class': 'label_final'}
# =====================================================================


def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


def load_train_test(repo_root, scheme):
    """Load train.csv & test.csv buat 1 skema, mapping label ke label_id (int), dan pastikan
    kolom 'fold' ada (string, blank buat baris non-asli)."""
    data_dir = repo_root / '03_Indobert' / 'Data' / scheme
    train_path = data_dir / 'train.csv'
    test_path = data_dir / 'test.csv'
    for p in (train_path, test_path):
        if not p.exists():
            raise FileNotFoundError(f'{p} tidak ditemukan. Jalankan gabung_final_training.py dulu.')

    train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False, encoding='utf-8')
    test_df = pd.read_csv(test_path, dtype=str, keep_default_na=False, encoding='utf-8')

    label_col = LABEL_COL[scheme]
    label2id = LABEL2ID[scheme]

    for nama, df in [('train', train_df), ('test', test_df)]:
        required = {'text', 'sumber', label_col}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f'{scheme}/{nama}.csv kurang kolom: {missing}')
        tidak_dikenal = set(df[label_col].unique()) - set(label2id.keys())
        if tidak_dikenal:
            raise ValueError(f'{scheme}/{nama}.csv punya label tak dikenal: {tidak_dikenal}')
        df['label_id'] = df[label_col].map(label2id)

    if 'fold' not in train_df.columns:
        train_df['fold'] = ''

    return train_df, test_df


def compute_class_weights(label_ids, n_classes):
    """Formula 'balanced' standar: weight_c = n_samples / (n_classes * count_c).
    Kelas yg jarang (misal negatif) dapet bobot loss lebih besar."""
    label_ids = np.asarray(label_ids)
    counts = np.array([(label_ids == c).sum() for c in range(n_classes)])
    if (counts == 0).any():
        raise ValueError(f'Ada kelas dengan 0 anggota: counts={counts.tolist()}')
    n_samples = len(label_ids)
    weights = n_samples / (n_classes * counts)
    return weights


def split_asli_only_validation(df, label_col='label_id', val_size=VAL_SIZE_INTERNAL, seed=42):
    """Pisahin validasi internal (buat early-stopping) HANYA dari baris asli; sisanya (asli
    yg ga kepilih + SEMUA sintesis/dummy) jadi training.

    Ini penting: kalau validasi ikut kecampur sintesis, itu bisa "bocor" -- sintesis adalah
    5 variasi paraphrase dari komentar YANG SAMA, jadi kalau 1 variasi masuk training dan
    variasi lain (dari komentar asal yang sama) masuk validasi, model keliatan jago pas
    divalidasi padahal cuma hafalan pola kalimat, bukan belajar sentimen beneran. Ini bikin
    early-stopping salah berhenti (telat), model jadi overfit ke training set (termasuk ke
    sintesis-nya sendiri) dan jeblok pas ketemu data asli yang beneran baru (test.csv).

    Fallback ke stratified split biasa kalau df ga punya baris asli sama sekali (jarang,
    misal potongan data residual di dalam 1 fold CV yang kebetulan abis)."""
    asli_mask = df['sumber'] == 'asli'
    asli_df = df[asli_mask]
    non_asli_df = df[~asli_mask]

    if len(asli_df) == 0:
        train_final_df, val_final_df = train_test_split(
            df, test_size=val_size, random_state=seed, stratify=df[label_col],
        )
        return train_final_df.reset_index(drop=True), val_final_df.reset_index(drop=True)

    try:
        asli_train_df, asli_val_df = train_test_split(
            asli_df, test_size=val_size, random_state=seed, stratify=asli_df[label_col],
        )
    except ValueError:
        # Baris asli kependekan buat stratified split di val_size ini (misal 1 kelas kurang
        # dari jumlah baris validasi yg diminta) -- fallback ke split non-stratified.
        asli_train_df, asli_val_df = train_test_split(
            asli_df, test_size=val_size, random_state=seed,
        )
    train_final_df = pd.concat([asli_train_df, non_asli_df], ignore_index=True)
    val_final_df = asli_val_df.reset_index(drop=True)
    return train_final_df, val_final_df


def split_fold_for_cv(df, fold_value, label_col='label_id', seed=42, val_size=VAL_SIZE_INTERNAL):
    """1 iterasi CV: pisahin held-out validation (cuma baris asli dgn fold == fold_value) dari
    sisanya (baris asli fold lain + SEMUA sintesis/dummy). Sisanya itu lalu dipecah lagi jadi
    train_sub_df/val_internal_df buat early-stopping -- val_internal_df-nya JUGA cuma dari
    baris asli (lihat split_asli_only_validation), biar early-stopping ga "bocor" ke sintesis.

    Return: (train_sub_df, val_internal_df, held_out_df)
    """
    held_out_mask = (df['sumber'] == 'asli') & (df['fold'] == str(fold_value))
    held_out_df = df[held_out_mask].reset_index(drop=True)
    remain_df = df[~held_out_mask].reset_index(drop=True)

    if len(held_out_df) == 0:
        raise ValueError(f'Ga ada baris asli dengan fold == {fold_value}. Cek kolom fold di train.csv.')

    train_sub_df, val_internal_df = split_asli_only_validation(
        remain_df, label_col=label_col, val_size=val_size, seed=seed,
    )
    return train_sub_df, val_internal_df, held_out_df


def available_folds(train_df):
    """Daftar nilai fold yg valid (cuma dari baris asli, exclude blank)."""
    folds = train_df.loc[train_df['sumber'] == 'asli', 'fold']
    folds = sorted({f for f in folds if f.strip() != ''}, key=int)
    return folds


# ====================== Bagian yang butuh torch/transformers ======================
# Import di sini (bukan di atas) biar logic di atas tetap bisa ditest tanpa torch terinstall.
def _lazy_import_ml():
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback,
        Trainer, TrainingArguments,
    )
    return torch, Dataset, AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback, Trainer, TrainingArguments


def build_dataset_class():
    torch, Dataset, *_ = _lazy_import_ml()

    class CommentDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len):
            enc = tokenizer(
                list(texts), truncation=True, padding='max_length',
                max_length=max_len, return_tensors='pt',
            )
            self.input_ids = enc['input_ids']
            self.attention_mask = enc['attention_mask']
            self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return {
                'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'labels': self.labels[idx],
            }

    return CommentDataset


def build_weighted_trainer_class():
    torch, _, _, _, _, Trainer, _ = _lazy_import_ml()

    class WeightedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            logits = outputs.logits
            weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
            loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    return WeightedTrainer


def make_compute_metrics(label_names):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        out = {
            'accuracy': accuracy_score(labels, preds),
            'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        }
        per_class = f1_score(labels, preds, average=None, zero_division=0,
                              labels=list(range(len(label_names))))
        for name, f1_val in zip(label_names, per_class):
            out[f'f1_{name}'] = f1_val
        return out
    return compute_metrics


def train_one_model(train_sub_df, val_df, label_names, class_weight_vec, tokenizer,
                     model_name, max_len, out_dir, seed_offset=0):
    """Latih 1 model IndoBERTweet. Return (trainer, model) biar bisa dipakai predict lagi."""
    torch, _, AutoModelForSequenceClassification, _, EarlyStoppingCallback, _, TrainingArguments = _lazy_import_ml()
    CommentDataset = build_dataset_class()
    WeightedTrainer = build_weighted_trainer_class()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_labels = len(label_names)

    train_ds = CommentDataset(train_sub_df['text'], train_sub_df['label_id'], tokenizer, max_len)
    val_ds = CommentDataset(val_df['text'], val_df['label_id'], tokenizer, max_len)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_labels)

    total_steps = (len(train_sub_df) // BATCH_TRAIN + 1) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    args = TrainingArguments(
        output_dir=str(out_dir / 'checkpoints'),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        fp16=(device == 'cuda'),
        logging_steps=50,
        seed=SEED + seed_offset,
        report_to='none',
        disable_tqdm=False,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=make_compute_metrics(label_names),
        class_weights=class_weight_vec,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
    )
    trainer.train()
    return trainer


def evaluate_on(trainer, eval_df, label_names, tokenizer, max_len):
    CommentDataset = build_dataset_class()
    ds = CommentDataset(eval_df['text'], eval_df['label_id'], tokenizer, max_len)
    pred_out = trainer.predict(ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    labels_true = eval_df['label_id'].astype(int).to_numpy()
    report = classification_report(labels_true, preds, target_names=label_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(labels_true, preds, labels=list(range(len(label_names))))
    return report, cm.tolist()


def main():
    torch, *_ = _lazy_import_ml()

    REPO_ROOT = find_repo_root()
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
            "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
        )

    MODELS_DIR = REPO_ROOT / '03_Indobert' / 'models'
    RESULTS_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_cv'
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:', device)
    if device == 'cuda':
        print('GPU   :', torch.cuda.get_device_name(0))
    else:
        print('PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.')

    _, _, AutoModelForSequenceClassification, AutoTokenizer, _, _, _ = _lazy_import_ml()

    all_fold_results = []
    all_fold_reports = {}
    final_results = {}

    for scheme in SCHEMES:
        label2id = LABEL2ID[scheme]
        label_names = [name for name, _ in sorted(label2id.items(), key=lambda kv: kv[1])]
        n_labels = len(label_names)

        train_df, test_df = load_train_test(REPO_ROOT, scheme)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        print(f'\n{"=" * 70}\n  SKEMA: {scheme}  ({n_labels} kelas)\n{"=" * 70}')
        print(f'  train total: {len(train_df)}  (asli={len(train_df[train_df.sumber == "asli"])}, '
              f'sintesis/dummy={len(train_df[train_df.sumber != "asli"])})')
        print(f'  test total : {len(test_df)}  (semua asli, hold-out sungguhan)')

        folds = available_folds(train_df)
        print(f'  fold tersedia (dari baris asli): {folds}')

        # ------------------------- CV (buat development/tuning) -------------------------
        if not RUN_CV:
            print('  RUN_CV=False -> skip 5-fold CV, langsung ke model final.')
        for fold in (folds if RUN_CV else []):
            t0 = time.time()
            train_sub_df, val_internal_df, held_out_df = split_fold_for_cv(
                train_df, fold, seed=SEED + int(fold),
            )
            class_weight_vec_np = compute_class_weights(train_sub_df['label_id'], n_labels)
            class_weight_vec = torch.tensor(class_weight_vec_np, dtype=torch.float)

            out_dir = MODELS_DIR / scheme / f'cv_fold_{fold}'
            trainer = train_one_model(
                train_sub_df, val_internal_df, label_names, class_weight_vec, tokenizer,
                MODEL_NAME, MAX_LEN, out_dir, seed_offset=int(fold),
            )

            report, cm = evaluate_on(trainer, held_out_df, label_names, tokenizer, MAX_LEN)
            row = {
                'scheme': scheme, 'fold': fold,
                'n_train': len(train_sub_df), 'n_val_internal': len(val_internal_df),
                'n_held_out_asli': len(held_out_df),
                'accuracy': report['accuracy'], 'f1_macro': report['macro avg']['f1-score'],
            }
            for name in label_names:
                row[f'f1_{name}'] = report[name]['f1-score']
                row[f'precision_{name}'] = report[name]['precision']
                row[f'recall_{name}'] = report[name]['recall']
            all_fold_results.append(row)
            all_fold_reports[(scheme, fold)] = {
                'report': report, 'confusion_matrix': cm, 'label_names': label_names,
            }
            print(f'  [fold {fold}] f1_macro={row["f1_macro"]:.3f} | '
                  + ' | '.join(f'f1_{n}={row[f"f1_{n}"]:.3f}' for n in label_names)
                  + f' | waktu={(time.time() - t0) / 60:.1f} menit')

            if SAVE_MODELS:
                out_dir.mkdir(parents=True, exist_ok=True)
                trainer.save_model(str(out_dir))
                tokenizer.save_pretrained(str(out_dir))

            del trainer
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

        # ------------------------- Model FINAL (evaluasi sekali ke test.csv) -------------------------
        print(f'\n  --- Melatih model FINAL {scheme} (seluruh train.csv) ---')
        class_weight_vec_np = compute_class_weights(train_df['label_id'], n_labels)
        class_weight_vec = torch.tensor(class_weight_vec_np, dtype=torch.float)

        train_sub_df, val_internal_df = split_asli_only_validation(
            train_df, label_col='label_id', val_size=VAL_SIZE_INTERNAL, seed=SEED,
        )
        out_dir_final = MODELS_DIR / scheme / 'final'
        trainer_final = train_one_model(
            train_sub_df, val_internal_df,
            label_names, class_weight_vec, tokenizer, MODEL_NAME, MAX_LEN, out_dir_final,
        )
        report_test, cm_test = evaluate_on(trainer_final, test_df, label_names, tokenizer, MAX_LEN)
        print(f'  [FINAL -> test.csv] accuracy={report_test["accuracy"]:.3f} '
              f'f1_macro={report_test["macro avg"]["f1-score"]:.3f}')
        for name in label_names:
            print(f'    f1_{name} = {report_test[name]["f1-score"]:.3f}')

        # Evaluasi juga ke TRAIN penuh -- buat bandingin train vs test (cek overfitting:
        # kalau train jauh lebih tinggi drpd test, model ngapalin training set).
        print(f'  --- Evaluasi model final ke TRAIN penuh (cek overfitting) ---')
        report_train, cm_train = evaluate_on(trainer_final, train_df, label_names, tokenizer, MAX_LEN)
        print(f'  [FINAL -> train.csv] accuracy={report_train["accuracy"]:.3f} '
              f'f1_macro={report_train["macro avg"]["f1-score"]:.3f}')

        final_results[scheme] = {
            'label_names': label_names,
            'train': {'report': report_train, 'confusion_matrix': cm_train},
            'test': {'report': report_test, 'confusion_matrix': cm_test},
        }

        if SAVE_MODELS:
            out_dir_final.mkdir(parents=True, exist_ok=True)
            trainer_final.save_model(str(out_dir_final))
            tokenizer.save_pretrained(str(out_dir_final))

        del trainer_final, tokenizer
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

    # ------------------------- Simpan hasil -------------------------
    if all_fold_results:
        results_df = pd.DataFrame(all_fold_results)
        results_df.to_csv(RESULTS_DIR / 'cv_results_per_fold.csv', index=False)

        metric_cols = [c for c in results_df.columns if c.startswith(('accuracy', 'f1_', 'precision_', 'recall_'))]
        summary = results_df.groupby('scheme')[metric_cols].agg(['mean', 'std'])
        summary.to_csv(RESULTS_DIR / 'cv_results_summary.csv')
    else:
        print('RUN_CV=False -> ga ada hasil CV yg disimpan (cv_results_per_fold.csv/summary.csv dilewat).')

    with open(RESULTS_DIR / 'cv_reports_detail.json', 'w', encoding='utf-8') as f:
        json.dump({f'{s}_fold{i}': v for (s, i), v in all_fold_reports.items()}, f, indent=2, ensure_ascii=False)

    with open(RESULTS_DIR / 'final_train_vs_test.json', 'w', encoding='utf-8') as f:
        json.dump(final_results, f, indent=2, ensure_ascii=False)

    print('\nTersimpan di 03_Indobert/Data/hasil_cv/:')
    print(' - cv_results_per_fold.csv     (metrik CV per fold, held-out cuma dari baris asli)')
    print(' - cv_results_summary.csv      (mean ± std per skema, dari CV)')
    print(' - cv_reports_detail.json      (classification_report + confusion matrix lengkap per fold)')
    print(' - final_train_vs_test.json    (HASIL AKHIR: model full-train dievaluasi ke train.csv DAN test.csv,')
    print('                                buat dibandingin lewat tampilkan_laporan_perbandingan.py)')


if __name__ == '__main__':
    main()

c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
PERHATIAN: tidak terdeteksi GPU. Training akan sangat lambat di CPU.

  SKEMA: binary  (2 kelas)
  train total: 2218  (asli=1668, sintesis/dummy=550)
  test total : 417  (semua asli, hold-out sungguhan)
  fold tersedia (dari baris asli): ['1', '2', '3', '4', '5']
  RUN_CV=False -> skip 5-fold CV, langsung ke model final.

  --- Melatih model FINAL binary (seluruh train.csv) ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17995.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Non-negatif,F1 Negatif
1,0.388925,0.260670,0.916168,0.685753,0.954839,0.416667
2,0.121512,0.236072,0.946108,0.818675,0.970684,0.666667
3,0.050009,0.361175,0.916168,0.758071,0.953642,0.562500
4,0.021326,0.325548,0.928144,0.780263,0.960526,0.600000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.23it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

  [FINAL -> test.csv] accuracy=0.923 f1_macro=0.704
    f1_non-negatif = 0.959
    f1_negatif = 0.448
  --- Evaluasi model final ke TRAIN penuh (cek overfitting) ---


... (4 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.54it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]
c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

  [FINAL -> test.csv] accuracy=0.851 f1_macro=0.657
    f1_negatif = 0.393
    f1_netral = 0.911
    f1_positif = 0.667
  --- Evaluasi model final ke TRAIN penuh (cek overfitting) ---


c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [FINAL -> train.csv] accuracy=0.988 f1_macro=0.988
RUN_CV=False -> ga ada hasil CV yg disimpan (cv_results_per_fold.csv/summary.csv dilewat).

Tersimpan di 03_Indobert/Data/hasil_cv/:
 - cv_results_per_fold.csv     (metrik CV per fold, held-out cuma dari baris asli)
 - cv_results_summary.csv      (mean ± std per skema, dari CV)
 - cv_reports_detail.json      (classification_report + confusion matrix lengkap per fold)
 - final_train_vs_test.json    (HASIL AKHIR: model full-train dievaluasi ke train.csv DAN test.csv,
                                buat dibandingin lewat tampilkan_laporan_perbandingan.py)


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Test logic finetune_cv_final.py yg GA butuh torch/transformers (fold splitting,
class weight, label mapping). Dijalankan terpisah dari training beneran."""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import finetune_cv_final as f  # noqa: E402

errors = []


def check(name, cond):
    status = 'OK' if cond else 'GAGAL'
    print(f'[{status}] {name}')
    if not cond:
        errors.append(name)


# --- Fixture: train_df kecil, 2 fold, campur asli + sintesis + dummy ---
rows = []
for fold in ['1', '2']:
    for i in range(6):
        rows.append({'text': f'netral asli f{fold}_{i}', 'label_id': 0, 'sumber': 'asli', 'fold': fold})
    for i in range(3):
        rows.append({'text': f'negatif asli f{fold}_{i}', 'label_id': 1, 'sumber': 'asli', 'fold': fold})
train_df = pd.DataFrame(rows)
# tambah sintesis & dummy negatif, TANPA fold
for i in range(10):
    train_df = pd.concat([train_df, pd.DataFrame([{
        'text': f'sintesis negatif {i}', 'label_id': 1, 'sumber': 'sintesis_paraphrase', 'fold': '',
    }])], ignore_index=True)
for i in range(3):
    train_df = pd.concat([train_df, pd.DataFrame([{
        'text': f'dummy negatif {i}', 'label_id': 1, 'sumber': 'dummy_baru', 'fold': '',
    }])], ignore_index=True)

# --- Test available_folds ---
folds = f.available_folds(train_df)
check('available_folds mengembalikan [1, 2]', folds == ['1', '2'])

# --- Test split_fold_for_cv: held-out cuma dari asli fold tsb ---
train_sub, val_internal, held_out = f.split_fold_for_cv(train_df, '1', seed=0)
check('held_out semua sumber == asli', (held_out['sumber'] == 'asli').all())
check('held_out semua fold == 1', (held_out['fold'] == '1').all())
check('held_out jumlahnya 9 (6 netral + 3 negatif fold 1)', len(held_out) == 9)
gabungan_train = pd.concat([train_sub, val_internal])
check('tidak ada baris held-out yg nyasar ke train_sub/val_internal',
      not gabungan_train['text'].isin(held_out['text']).any())
check('semua baris sintesis/dummy (fold kosong) masuk ke train_sub atau val_internal, bukan held_out',
      not held_out['sumber'].isin(['sintesis_paraphrase', 'dummy_baru']).any())
total_cek = len(train_sub) + len(val_internal) + len(held_out)
check('total baris balance (train_sub + val_internal + held_out == total awal)', total_cek == len(train_df))

# --- Test split_fold_for_cv fold lain (2) ---
train_sub2, val_internal2, held_out2 = f.split_fold_for_cv(train_df, '2', seed=0)
check('held_out fold 2 beda sama held_out fold 1 (ga overlap)',
      not held_out2['text'].isin(held_out['text']).any())

# --- Test error kalau fold ga ada ---
try:
    f.split_fold_for_cv(train_df, '99', seed=0)
    check('split_fold_for_cv raise error kalau fold ga ada', False)
except ValueError:
    check('split_fold_for_cv raise error kalau fold ga ada', True)

# --- Test compute_class_weights ---
# label_id: kelas 0 ada 20 (2 fold x (6+10sintesis dinilai fold label beda)), hitung manual langsung dari train_sub
labels_contoh = np.array([0] * 18 + [1] * 2)  # imbalance kuat
w = f.compute_class_weights(labels_contoh, 2)
n = len(labels_contoh)
expected_w0 = n / (2 * 18)
expected_w1 = n / (2 * 2)
check('compute_class_weights kelas minoritas dapet bobot lebih besar', w[1] > w[0])
check('compute_class_weights nilai sesuai formula balanced',
      abs(w[0] - expected_w0) < 1e-9 and abs(w[1] - expected_w1) < 1e-9)

try:
    f.compute_class_weights(np.array([0, 0, 0]), 2)
    check('compute_class_weights raise error kalau ada kelas 0 anggota', False)
except ValueError:
    check('compute_class_weights raise error kalau ada kelas 0 anggota', True)

# --- Test load_train_test: kolom & label mapping ---
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    (base / '01_pengumpulan_data').mkdir()
    (base / '03_Indobert' / 'Data' / 'binary').mkdir(parents=True)
    (base / '03_Indobert' / 'Data' / '3class').mkdir(parents=True)

    df_bin_train = pd.DataFrame({
        'text': ['komentar a', 'komentar b', 'komentar c'],
        'sumber': ['asli', 'asli', 'sintesis_paraphrase'],
        'label_name': ['non-negatif', 'negatif', 'negatif'],
        'fold': ['1', '2', ''],
    })
    df_bin_test = pd.DataFrame({
        'text': ['komentar d'], 'sumber': ['asli'], 'label_name': ['non-negatif'],
    })
    df_bin_train.to_csv(base / '03_Indobert' / 'Data' / 'binary' / 'train.csv', index=False)
    df_bin_test.to_csv(base / '03_Indobert' / 'Data' / 'binary' / 'test.csv', index=False)

    train_loaded, test_loaded = f.load_train_test(base, 'binary')
    check('load_train_test mapping label_id benar (non-negatif=0, negatif=1)',
          list(train_loaded['label_id']) == [0, 1, 1])
    check('load_train_test kolom fold otomatis ada di train (test.csv ga perlu fold)',
          'fold' in train_loaded.columns)

    # test label tak dikenal -> error
    df_bin_bad = df_bin_train.copy()
    df_bin_bad.loc[0, 'label_name'] = 'label_aneh'
    df_bin_bad.to_csv(base / '03_Indobert' / 'Data' / 'binary' / 'train.csv', index=False)
    try:
        f.load_train_test(base, 'binary')
        check('load_train_test raise error kalau ada label tak dikenal', False)
    except ValueError:
        check('load_train_test raise error kalau ada label tak dikenal', True)

print()
if errors:
    print(f'{len(errors)} TEST GAGAL: {errors}')
    sys.exit(1)
else:
    print('SEMUA TEST LOGIC LOLOS (tanpa butuh torch/transformers/GPU).')


[OK] available_folds mengembalikan [1, 2]
[OK] held_out semua sumber == asli
[OK] held_out semua fold == 1
[OK] held_out jumlahnya 9 (6 netral + 3 negatif fold 1)
[OK] tidak ada baris held-out yg nyasar ke train_sub/val_internal
[OK] semua baris sintesis/dummy (fold kosong) masuk ke train_sub atau val_internal, bukan held_out
[OK] total baris balance (train_sub + val_internal + held_out == total awal)
[OK] held_out fold 2 beda sama held_out fold 1 (ga overlap)
[OK] split_fold_for_cv raise error kalau fold ga ada
[OK] compute_class_weights kelas minoritas dapet bobot lebih besar
[OK] compute_class_weights nilai sesuai formula balanced
[OK] compute_class_weights raise error kalau ada kelas 0 anggota
[OK] load_train_test mapping label_id benar (non-negatif=0, negatif=1)
[OK] load_train_test kolom fold otomatis ada di train (test.csv ga perlu fold)
[OK] load_train_test raise error kalau ada label tak dikenal

SEMUA TEST LOGIC LOLOS (tanpa butuh torch/transformers/GPU).


In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Tampilin laporan hasil evaluasi IndoBERTweet secara rapi, dari file
`03_Indobert/Data/hasil_cv/final_train_vs_test.json` (hasil finetune_cv_final.py).

Nampilin buat tiap skema (binary & 3class):
  1. Tabel precision/recall/F1-score/support per kelas (ini yg dimaksud "kaya confusion
     matrix tapi ada precision/recall/F1" -- fungsi python-nya classification_report dari
     sklearn, di sini cuma dirapiin jadi tabel) -- SISI BERDAMPINGAN train vs test, biar
     langsung kelihatan beda performanya.
  2. Confusion matrix asli (jumlah prediksi x label asli) buat train & test.
  3. Ringkasan gap accuracy/F1-macro train vs test -- gap besar = sinyal overfitting.

Bisa dipanggil langsung (baca dari path default) atau import function-nya buat dipakai di
notebook/script lain.
"""
import json
from pathlib import Path

import pandas as pd


def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


def muat_hasil(path):
    if not path.exists():
        raise FileNotFoundError(
            f'{path} tidak ditemukan. Jalankan finetune_cv_final.py dulu sampai selesai.'
        )
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def tabel_classification_report(report, label_names):
    """report = dict hasil sklearn.metrics.classification_report(..., output_dict=True).
    Dirapiin jadi DataFrame: baris = kelas (+ accuracy/macro avg/weighted avg),
    kolom = precision/recall/f1-score/support."""
    baris = []
    for name in label_names:
        r = report[name]
        baris.append({
            'kelas': name,
            'precision': r['precision'],
            'recall': r['recall'],
            'f1-score': r['f1-score'],
            'support': int(r['support']),
        })
    for ringkasan_key, label_tampil in [('macro avg', 'macro avg'), ('weighted avg', 'weighted avg')]:
        r = report[ringkasan_key]
        baris.append({
            'kelas': label_tampil,
            'precision': r['precision'],
            'recall': r['recall'],
            'f1-score': r['f1-score'],
            'support': int(r['support']),
        })
    baris.append({
        'kelas': 'accuracy', 'precision': '', 'recall': '', 'f1-score': report['accuracy'],
        'support': int(report['macro avg']['support']),
    })
    df = pd.DataFrame(baris).set_index('kelas')
    return df


def tabel_perbandingan_train_test(report_train, report_test, label_names):
    """Precision/recall/F1 per kelas, train & test berdampingan, plus kolom gap (test - train)
    buat F1 -- gap negatif besar = overfitting (test jauh lebih jelek drpd train)."""
    baris = []
    for name in label_names + ['macro avg', 'weighted avg']:
        rt, rv = report_train[name], report_test[name]
        baris.append({
            'kelas': name,
            'precision_train': rt['precision'], 'precision_test': rv['precision'],
            'recall_train': rt['recall'], 'recall_test': rv['recall'],
            'f1_train': rt['f1-score'], 'f1_test': rv['f1-score'],
            'gap_f1 (test - train)': rv['f1-score'] - rt['f1-score'],
        })
    baris.append({
        'kelas': 'accuracy',
        'precision_train': '', 'precision_test': '', 'recall_train': '', 'recall_test': '',
        'f1_train': report_train['accuracy'], 'f1_test': report_test['accuracy'],
        'gap_f1 (test - train)': report_test['accuracy'] - report_train['accuracy'],
    })
    df = pd.DataFrame(baris).set_index('kelas')
    return df


def tabel_confusion_matrix(cm, label_names):
    """cm = list of list (hasil sklearn confusion_matrix). Baris = label asli, kolom = prediksi."""
    idx = [f'asli: {n}' for n in label_names]
    cols = [f'prediksi: {n}' for n in label_names]
    return pd.DataFrame(cm, index=idx, columns=cols)


def cetak_laporan_skema(scheme, data, float_fmt='{:.3f}'):
    label_names = data['label_names']
    report_train = data['train']['report']
    report_test = data['test']['report']
    cm_train = data['train']['confusion_matrix']
    cm_test = data['test']['confusion_matrix']

    pd.set_option('display.float_format', float_fmt.format)

    print('\n' + '=' * 70)
    print(f'SKEMA: {scheme.upper()}')
    print('=' * 70)

    print('\n-- Classification report TRAIN --')
    print(tabel_classification_report(report_train, label_names).to_string())

    print('\n-- Classification report TEST --')
    print(tabel_classification_report(report_test, label_names).to_string())

    print('\n-- Perbandingan TRAIN vs TEST (gap F1 negatif besar = overfitting) --')
    print(tabel_perbandingan_train_test(report_train, report_test, label_names).to_string())

    print('\n-- Confusion matrix TRAIN --')
    print(tabel_confusion_matrix(cm_train, label_names).to_string())

    print('\n-- Confusion matrix TEST --')
    print(tabel_confusion_matrix(cm_test, label_names).to_string())

    gap_acc = report_test['accuracy'] - report_train['accuracy']
    gap_f1 = report_test['macro avg']['f1-score'] - report_train['macro avg']['f1-score']
    print(f'\nGap accuracy (test - train): {gap_acc:+.3f}')
    print(f'Gap F1-macro (test - train): {gap_f1:+.3f}')
    if gap_f1 < -0.1:
        print('  -> PERHATIAN: gap > 0.1, kemungkinan model overfit ke training set.')
    else:
        print('  -> gap wajar, model keliatan generalize dengan oke.')


def main():
    REPO_ROOT = find_repo_root()
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
            "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
        )
    path = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_cv' / 'final_train_vs_test.json'
    hasil = muat_hasil(path)
    for scheme, data in hasil.items():
        cetak_laporan_skema(scheme, data)


if __name__ == '__main__':
    main()


SKEMA: BINARY

-- Classification report TRAIN --
             precision recall  f1-score  support
kelas                                           
non-negatif      0.995  0.985     0.990     1564
negatif          0.966  0.989     0.977      654
macro avg        0.981  0.987     0.984     2218
weighted avg     0.987  0.986     0.987     2218
accuracy                          0.986     2218

-- Classification report TEST --
             precision recall  f1-score  support
kelas                                           
non-negatif      0.966  0.951     0.959      391
negatif          0.406  0.500     0.448       26
macro avg        0.686  0.726     0.704      417
weighted avg     0.931  0.923     0.927      417
accuracy                          0.923      417

-- Perbandingan TRAIN vs TEST (gap F1 negatif besar = overfitting) --
             precision_train precision_test recall_train recall_test  f1_train  f1_test  gap_f1 (test - train)
kelas                                           